<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-03-prompting/lesson-3.3-model-routing/notebooks/GCP_Capstone_3.3_Model_Routing.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.3 Chain-of-Thought & Model Routing
**Netsetos GenAI Engineering — GCP Capstone**

Thinking mode, complexity classifier, model router, 84% cost savings.


## Setup


In [ ]:
!pip install -q google-genai==2.21.0 pydantic
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE THIS

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import Literal
import json, time

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')


In [ ]:
import json
from pydantic import ValidationError

def draft_of(r, schema, quote_limit=200):
    """The model's structured answer, or a clear error - never a silent None.

    The SDK sets r.parsed to None on ANY validation failure. The first live run of the lane
    (7 Sept 2026) met the one that matters: a quote longer than the contract's 200 characters -
    a statute provision is one sentence - which threw fourteen right answers away and, worse,
    had been scored as the model refusing. So: read the JSON, trim the quote to the contract,
    validate again; anything else is an error you can read, not a refusal."""
    if r.parsed is not None:
        return r.parsed if isinstance(r.parsed, schema) else schema.model_validate(r.parsed)
    cand = (r.candidates or [None])[0]
    reason = getattr(getattr(cand, "finish_reason", None), "name", "NO_CANDIDATES")
    try:
        obj = json.loads(r.text or "")
    except ValueError:
        raise RuntimeError(f"no JSON to parse (finish_reason={reason}) - raise max_output_tokens if it is MAX_TOKENS")
    if isinstance(obj, dict):
        for c in obj.get("citations") or []:
            if isinstance(c, dict) and isinstance(c.get("quote"), str) and len(c["quote"]) > quote_limit:
                c["quote"] = c["quote"][: quote_limit - 3].rstrip() + "..."
    try:
        return schema.model_validate(obj)
    except ValidationError as e:
        err = e.errors()[0]
        raise RuntimeError(f"the draft failed the contract at {'.'.join(str(x) for x in err['loc'])}: {err['msg']}") from None


## Cell 1: Thinking Mode — Inspect Thought Tokens


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What is 17 * 23 + 45 * 12?',
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_level="MEDIUM", include_thoughts=True)),
)

for part in response.candidates[0].content.parts:
    if not part.text: continue
    label = 'THOUGHT' if part.thought else 'ANSWER'
    print(f'{label}: {part.text[:100]}')

meta = response.usage_metadata
print(f'\nInput: {meta.prompt_token_count} | Output: {meta.candidates_token_count} | Thinking: {meta.thoughts_token_count}')


## Cell 2: Thinking Level Comparison


In [ ]:
question = 'What is 17 * 23 + 456 - 89?'

for level in ["LOW", "MEDIUM", "HIGH"]:   # Gemini 3.x thinks at a LEVEL, never off
    r = client.models.generate_content(
        model='gemini-3.6-flash', contents=question,
        config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_level=level)))
    think = r.usage_metadata.thoughts_token_count or 0
    out = r.usage_metadata.candidates_token_count or 0
    print(f'  level={level:<6} think={think:<5} out={out:<5} answer={(r.text or "").strip()[:40]}')


## Cell 3: Structured CoT — Reasoning Before Answer


In [ ]:
class ReasonedAnswer(BaseModel):
    reasoning: str = Field(description='Step-by-step reasoning')
    answer: str = Field(description='Final answer')
    confidence: Literal['high','medium','low']

r = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='A train travels 120km in 1.5 hours. Average speed?',
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=ReasonedAnswer,
        # No temperature/top_p/top_k on Gemini 3.x: 3.6 Flash ignores them; on Gemini 3.1 Pro (Preview) / Flash-Lite Google says leave the default (verified 2026-09-03)
        thinking_config=types.ThinkingConfig(thinking_level="LOW")))

result = draft_of(r, ReasonedAnswer)
print(f'Reasoning: {result.reasoning}')
print(f'Answer: {result.answer}')
print(f'Confidence: {result.confidence}')


## Cell 4: Complexity Classifier


In [ ]:
def classify_complexity(query):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f"""Classify complexity:
SIMPLE: Factual lookups, greetings, definitions, classification
MEDIUM: Explanations, comparisons, summaries, standard code
COMPLEX: Multi-step math, architecture, debugging, proofs

Query: {query}""",
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema={'type':'STRING','enum':['SIMPLE','MEDIUM','COMPLEX']},
            thinking_config=types.ThinkingConfig(thinking_level="LOW")))
    return r.text

test_queries = [
    'What is Python?',
    'Compare REST vs GraphQL for microservices',
    'Prove sqrt(2) is irrational',
    'What time is it?',
    'Design a distributed cache with consistency guarantees',
]
for q in test_queries:
    print(f'  {classify_complexity(q):<8} | {q}')


## Cell 5: Model Router


In [ ]:
# No temperature key: ignored on 3.6 Flash, leave the default elsewhere (F1).
# COMPLEX cap is 16384 because thinking tokens share max_output_tokens.
ROUTING_TABLE = {
    'SIMPLE':  {'model':'gemini-3.1-flash-lite','level':"LOW",      'max':512},
    'MEDIUM':  {'model':'gemini-3.6-flash',     'level':"MEDIUM",'max':2048},
    'COMPLEX': {'model':'gemini-3.1-pro-preview',       'level':"HIGH",  'max':16384},
}

def route_and_generate(query, system_instruction=''):
    complexity = classify_complexity(query)
    cfg = ROUTING_TABLE[complexity]
    r = client.models.generate_content(
        model=cfg['model'], contents=query,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            max_output_tokens=cfg['max'],
            thinking_config=types.ThinkingConfig(thinking_level=cfg['level'])))
    meta = r.usage_metadata
    return {'text':r.text, 'complexity':complexity, 'model':cfg['model'],
            'think_tokens':meta.thoughts_token_count or 0, 'out_tokens':meta.candidates_token_count}

for q in test_queries[:3]:
    result = route_and_generate(q)
    print(f'{result["complexity"]:<8} -> {result["model"]:<25} think={result["think_tokens"]}')
    print(f'  {(result["text"] or "")[:80]}...\n')


## Cell 6: Cost Calculator


In [ ]:
PRICING = {
    'gemini-3.1-flash-lite': {'input':0.25, 'output':1.50},
    'gemini-3.6-flash':      {'input':1.50, 'output':7.50},  # standard rate (from 2027-01-01); intro $0.75/$3.75 through 2026-12-31 - see lesson 1.3
    'gemini-3.1-pro-preview':        {'input':2.00, 'output':12.00},  # (Preview - re-verify GA status before v1.1 ships)
}

def cost_per_1k(dist):
    total = 0
    for tier, d in dist.items():
        n = d['queries']
        p = PRICING[d['model']]
        c = n*d['avg_in']/1e6*p['input'] + n*(d['avg_out']+d['avg_think'])/1e6*p['output']
        print(f'  {tier:<10} {n:>4}q | {d["model"]:<25} | ${c:.3f}')
        total += c
    print(f'  TOTAL: ${total:.3f}')
    return total

print('ROUTED (70/25/5):')
routed = cost_per_1k({
    'simple':  {'queries':700,'model':'gemini-3.1-flash-lite','avg_in':200,'avg_out':100,'avg_think':0},
    'medium':  {'queries':250,'model':'gemini-3.6-flash','avg_in':400,'avg_out':250,'avg_think':1024},
    'complex': {'queries':50, 'model':'gemini-3.1-pro-preview','avg_in':600,'avg_out':500,'avg_think':8192},
})
print('\nALWAYS-PRO:')
pro = cost_per_1k({
    'all': {'queries':1000,'model':'gemini-3.1-pro-preview','avg_in':300,'avg_out':200,'avg_think':4000},
})
print(f'\nSAVINGS: {(1-routed/pro)*100:.1f}% vs always-Pro')


## Cell 7: Cascade Router (Advanced)


In [ ]:
from pydantic import BaseModel

# Cascade router: try the cheapest model first, escalate only when it isn't confident.
# NOTE: token logprobs (the ideal confidence signal) are NOT exposed for Gemini 3.x on
# Vertex -- response_logprobs raises "Logprobs is not supported for this model"; they
# exist only on older 2.0-era models. So we cascade on the model's SELF-REPORTED
# confidence via structured output (a practical proxy -- calibrate thresholds on real data).

class ConfidentAnswer(BaseModel):
    answer: str
    confidence: float  # 0.0-1.0: how sure the answer is correct AND complete

def cascade_route(query):
    stages = [
        ('gemini-3.1-flash-lite', 'LOW'),      # cheapest, lowest thinking level
        ('gemini-3.6-flash', 'MEDIUM'),        # mid tier, some thinking
        ('gemini-3.1-pro-preview', 'HIGH'),  # last resort, deep thinking
    ]
    thresholds = [0.75, 0.60]  # escalate if self-confidence is below this
    for i, (model, level) in enumerate(stages):
        r = client.models.generate_content(
            model=model,
            contents='Answer the query, then set confidence (0-1) to how sure you '
                     'are your answer is correct and complete. Query: ' + query,
            config=types.GenerateContentConfig(
                thinking_config=types.ThinkingConfig(thinking_level=level),
                response_mime_type='application/json',
                response_schema=ConfidentAnswer))
        a = draft_of(r, ConfidentAnswer)
        if i == len(stages) - 1:
            return a.answer, model, 'last_resort'
        if a.confidence >= thresholds[i]:
            return a.answer, model, 'confident (%.2f)' % a.confidence
        print('  Escalating from %s (confidence=%.2f)' % (model, a.confidence))
    return a.answer, model, 'escalated'

for q in ['What is Python?', 'Prove sqrt(2) is irrational']:
    text, model, reason = cascade_route(q)
    print('%s (%s): %s...' % (model, reason, text[:60]))

## Cell 8: The model router module (the kit: services/rag-api/router.py)

In production this router lives at `deploy/services/rag-api/router.py` (Complexity enum SIMPLE / MEDIUM / COMPLEX, same routing table); lesson 12.6 wires it into the `/v1/query` handler and sets the per-tier quotas.


In [ ]:
# The lesson's router module - the lane's is deploy/services/rag-api/router.py, with the same two names: classify(), route()
def classify(query):
    r = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=f"""Classify this query's complexity level.

SIMPLE: Factual lookups, greetings, definitions, yes/no, classification, extraction
MEDIUM: Explanations, comparisons, summaries, standard code, multi-paragraph writing
COMPLEX: Multi-step math, architecture design, debugging, proofs, strategic reasoning

Query: {query}""",
        config=types.GenerateContentConfig(
            response_mime_type='text/x.enum',
            response_schema={'type':'STRING','enum':['SIMPLE','MEDIUM','COMPLEX']},
            thinking_config=types.ThinkingConfig(thinking_level="LOW")))
    return r.text

def route(query, system_instruction='', schema=None):
    complexity = classify(query)
    cfg = ROUTING_TABLE[complexity]
    config = types.GenerateContentConfig(
        system_instruction=system_instruction,
        max_output_tokens=cfg['max'],
        thinking_config=types.ThinkingConfig(thinking_level=cfg['level']))
    if schema:
        config.response_mime_type = 'application/json'
        config.response_schema = schema
    r = client.models.generate_content(model=cfg['model'], contents=query, config=config)
    return {'text':r.text, 'complexity':complexity, 'model':cfg['model'],
            'think_tokens':r.usage_metadata.thoughts_token_count or 0}

print('Module ready: classify(), route()')
print('Integration: route(query, system_instruction=PERSONA, schema=ModelDraft) - then resolve(draft, packed) -> RAGAnswer; never ask the model for RAGAnswer itself')


## ✅ Module 3 Complete!

- ✅ 3.1: system_instruction, ThinkingConfig, output caps, personas, A/B testing
- ✅ 3.2: JSON schema, Pydantic, enum, few-shot, anyOf
- ✅ 3.3: Thinking mode, structured CoT, complexity classifier, model router, 84% savings

**Three production modules:**
- The prompt config module - 4 preset configs + persona router (the lane's: deploy/services/rag-api/generator.py)
- The structured-output module - RAGAnswer, DocMetadata, config factories (the contract: deploy/shared/documind_schemas.py)
- The model router module - classify(), route(), cost tracking (the lane's: deploy/services/rag-api/router.py)

**Next: Module 4 — RAG (DIY → Managed):** Document AI ingestion (4.1) and the DIY pipeline (4.2) feed retrieved context into this routing layer; 4.5 budgets it.
